# WP2 — Baseline Fairness Audit
Measures disparate impact of the baseline model across demographic groups before mitigation.

**Run order:** parse_training_cvs.py → train.py → proxy_detection.py → this notebook → train_fair.py

In [ ]:
import sys
sys.path.insert(0, '/app/ml')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from fairlearn.metrics import (
    MetricFrame,
    selection_rate,
    true_positive_rate,
    false_positive_rate,
)
from sklearn.metrics import roc_auc_score

DATA_PATH  = Path('/app/ml/training_dataset.csv')
MODEL_PATH = Path('/app/ml/model.joblib')

FEATURE_COLUMNS = [
    'total_years_experience', 'num_positions', 'avg_tenure_months',
    'education_level_score', 'total_skills_count', 'has_certifications',
    'language_count', 'section_completeness_score', 'max_language_score',
    'has_senior_title', 'career_gap_months', 'latest_job_duration',
    'has_summary', 'num_certifications', 'parse_quality_score',
    'experience_education_ratio', 'certs_per_year',
    'experience_x_seniority', 'experience_x_education',
]

df = pd.read_csv(DATA_PATH)
print(f'Dataset: {len(df)} rows')
print(df[['gender','age_cohort','is_multilingual','passed_next_stage']].describe())

## 1. Load model and generate predictions

In [ ]:
artifact = joblib.load(MODEL_PATH)
pipeline = artifact['pipeline']
print(f"Model: {artifact['model_name']}")
print(f"AUC (train eval): {artifact['best_auc']:.3f}")

X = df[FEATURE_COLUMNS].fillna(0).values
y = df['passed_next_stage'].values

# Use same split as train.py for reproducibility
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, df.index, test_size=0.2, stratify=y, random_state=42
)

df_test = df.loc[idx_test].copy()
y_pred  = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

print(f'\nTest set: {len(X_test)} candidates')
print(f'Overall AUC: {roc_auc_score(y_test, y_proba):.3f}')

## 2. Fairness metrics — MetricFrame

**Primary:** Equal Opportunity (TPR equality) — are qualified candidates from all groups equally likely to be invited?  
**Secondary:** Demographic Parity (selection rate) — raw invite rate per group.  
**Tertiary:** False Positive Rate — are unqualified candidates equally wrongly invited?

In [ ]:
# ── Gender ────────────────────────────────────────────────────────────────────
mf_gender = MetricFrame(
    metrics={'selection_rate': selection_rate, 'tpr': true_positive_rate, 'fpr': false_positive_rate},
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=df_test['gender'],
)

print('=== Gender ===')
print(mf_gender.by_group.round(3))
print(f"\nEqual Opportunity Difference (TPR): {mf_gender.difference()['tpr']:.3f}")
print(f"Demographic Parity Difference:      {mf_gender.difference()['selection_rate']:.3f}")

In [ ]:
# ── Age cohort ────────────────────────────────────────────────────────────────
mf_age = MetricFrame(
    metrics={'selection_rate': selection_rate, 'tpr': true_positive_rate, 'fpr': false_positive_rate},
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=df_test['age_cohort'],
)

print('=== Age Cohort ===')
print(mf_age.by_group.round(3))
print(f"\nEqual Opportunity Difference (TPR): {mf_age.difference()['tpr']:.3f}")
print(f"Demographic Parity Difference:      {mf_age.difference()['selection_rate']:.3f}")

In [ ]:
# ── Multilingual ──────────────────────────────────────────────────────────────
mf_lang = MetricFrame(
    metrics={'selection_rate': selection_rate, 'tpr': true_positive_rate, 'fpr': false_positive_rate},
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=df_test['is_multilingual'].map({0: 'monolingual', 1: 'multilingual'}),
)

print('=== Multilingual ===')
print(mf_lang.by_group.round(3))
print(f"\nEqual Opportunity Difference (TPR): {mf_lang.difference()['tpr']:.3f}")
print(f"Demographic Parity Difference:      {mf_lang.difference()['selection_rate']:.3f}")

## 3. BCa Bootstrap Confidence Intervals
With subgroups of 30–80 samples, raw disparity numbers may be noise.  
Rule: **95% CI strictly above 0.05 → mitigation required.**

In [ ]:
from sklearn.utils import resample

N_BOOTSTRAP = 10_000
THRESHOLD   = 0.05

def bootstrap_eod(y_true, y_pred, sensitive, n=N_BOOTSTRAP, seed=42):
    """BCa bootstrap CI for Equal Opportunity Difference."""
    rng = np.random.default_rng(seed)
    diffs = []
    idx = np.arange(len(y_true))
    for _ in range(n):
        s = rng.choice(idx, size=len(idx), replace=True)
        mf = MetricFrame(
            metrics={'tpr': true_positive_rate},
            y_true=y_true[s],
            y_pred=y_pred[s],
            sensitive_features=sensitive.iloc[s] if hasattr(sensitive, 'iloc') else sensitive[s],
        )
        diffs.append(mf.difference()['tpr'])
    diffs = np.array(diffs)
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return float(np.mean(diffs)), lo, hi

y_test_arr   = np.array(y_test)
y_pred_arr   = np.array(y_pred)

print('BCa 95% CI for Equal Opportunity Difference (10 000 iterations)')
print(f"{'Attribute':<20} {'Mean':>8} {'CI_low':>8} {'CI_high':>8}  Verdict")
print('-' * 65)

for label, sf in [
    ('gender',       df_test['gender']),
    ('age_cohort',   df_test['age_cohort']),
    ('multilingual', df_test['is_multilingual'].map({0:'mono',1:'multi'})),
]:
    mean, lo, hi = bootstrap_eod(y_test_arr, y_pred_arr, sf)
    verdict = '⚠️  MITIGATE' if lo > THRESHOLD else ('✅ OK' if hi < THRESHOLD else '⚠️  BORDERLINE')
    print(f"  {label:<18} {mean:>8.3f} {lo:>8.3f} {hi:>8.3f}  {verdict}")

## 4. Visualisations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Baseline Model — Selection Rate by Demographic Group', fontsize=13)

for ax, (mf, title) in zip(axes, [
    (mf_gender, 'Gender'),
    (mf_age,    'Age Cohort'),
    (mf_lang,   'Language background'),
]):
    sr = mf.by_group['selection_rate']
    colors = ['#1d6feb' if v == sr.min() else '#6b9fd4' for v in sr]
    bars = ax.bar(sr.index, sr.values, color=colors, edgecolor='white', linewidth=0.5)
    ax.axhline(mf.overall['selection_rate'], color='red', linestyle='--', linewidth=1, label='Overall')
    ax.set_title(title)
    ax.set_ylabel('Selection rate')
    ax.set_ylim(0, 0.5)
    ax.tick_params(axis='x', rotation=15)
    ax.legend(fontsize=8)
    for bar, val in zip(bars, sr.values):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.01, f'{val:.2f}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('/app/ml/fairness_audit/baseline_selection_rate.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved.')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Baseline Model — True Positive Rate (Equal Opportunity) by Group', fontsize=13)

for ax, (mf, title) in zip(axes, [
    (mf_gender, 'Gender'),
    (mf_age,    'Age Cohort'),
    (mf_lang,   'Language background'),
]):
    tpr = mf.by_group['tpr']
    colors = ['#dc2626' if v == tpr.min() else '#f87171' for v in tpr]
    bars = ax.bar(tpr.index, tpr.values, color=colors, edgecolor='white', linewidth=0.5)
    ax.axhline(mf.overall['tpr'], color='black', linestyle='--', linewidth=1, label='Overall')
    ax.set_title(title)
    ax.set_ylabel('TPR (Equal Opportunity)')
    ax.set_ylim(0, 1.0)
    ax.tick_params(axis='x', rotation=15)
    ax.legend(fontsize=8)
    for bar, val in zip(bars, tpr.values):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.02, f'{val:.2f}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('/app/ml/fairness_audit/baseline_tpr.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved.')

## 5. Summary
Run `train_fair.py` next to produce the mitigated model, then re-run this notebook loading `model_fair.joblib` to compare.

## 6. Fair Model Comparison
Load `model_fair.joblib` (re-weighted LogisticRegression, WP2 Phase 3) and compare
side-by-side with the baseline on the same test split.

In [ ]:
FAIR_MODEL_PATH = Path('/app/ml/model_fair.joblib')

if not FAIR_MODEL_PATH.exists():
    print('model_fair.joblib not found — run train_fair.py first.')
else:
    fair_artifact = joblib.load(FAIR_MODEL_PATH)
    fair_pipeline = fair_artifact['pipeline']
    print(f"Fair model: {fair_artifact['model_name']}")
    print(f"AUC: {fair_artifact['best_auc']:.3f} | "
          f"Temperature: {fair_artifact.get('temperature', 'N/A')}")
    print(f"Labels cleaned: {fair_artifact.get('n_labels_cleaned', '?')}")
    print(f"Decision threshold: {fair_artifact.get('decision_threshold', 0.5):.2f}")

    y_pred_fair  = fair_pipeline.predict(X_test)
    y_proba_fair = fair_pipeline.predict_proba(X_test)[:, 1]
    print(f'\nFair model AUC on test set: {roc_auc_score(y_test, y_proba_fair):.3f}')

In [ ]:
# ── MetricFrame for fair model ────────────────────────────────────────────
mf_fair_gender = MetricFrame(
    metrics={'selection_rate': selection_rate, 'tpr': true_positive_rate, 'fpr': false_positive_rate},
    y_true=y_test, y_pred=y_pred_fair, sensitive_features=df_test['gender'],
)
mf_fair_age = MetricFrame(
    metrics={'selection_rate': selection_rate, 'tpr': true_positive_rate, 'fpr': false_positive_rate},
    y_true=y_test, y_pred=y_pred_fair, sensitive_features=df_test['age_cohort'],
)
mf_fair_lang = MetricFrame(
    metrics={'selection_rate': selection_rate, 'tpr': true_positive_rate, 'fpr': false_positive_rate},
    y_true=y_test, y_pred=y_pred_fair,
    sensitive_features=df_test['is_multilingual'].map({0: 'monolingual', 1: 'multilingual'}),
)

for attr, mf_base, mf_fair in [
    ('gender',       mf_gender, mf_fair_gender),
    ('age_cohort',   mf_age,    mf_fair_age),
    ('multilingual', mf_lang,   mf_fair_lang),
]:
    b_eod = mf_base.difference()['tpr']
    f_eod = mf_fair.difference()['tpr']
    delta = f_eod - b_eod
    flag  = '✅' if abs(f_eod) <= abs(b_eod) else '⚠️'
    print(f"{attr:<18} baseline EOD={b_eod:+.3f}  fair EOD={f_eod:+.3f}  delta={delta:+.3f}  {flag}")

In [ ]:
# ── Bootstrap CI comparison ────────────────────────────────────────────────
import sys
sys.path.insert(0, '/app/ml')
from fairness_audit.bootstrap_ci import bca_eod_ci

print('BCa 95% Bootstrap CI comparison (n=10 000 iterations)\n')
print(f"{'Attribute':<18} {'Model':<10} {'EOD':>7} {'CI_low':>8} {'CI_high':>8}  Verdict")
print('-' * 70)

sensitive_map = [
    ('gender',       df_test['gender'].values),
    ('age_cohort',   df_test['age_cohort'].values),
    ('multilingual', df_test['is_multilingual'].map({0:'mono',1:'multi'}).values),
]

for attr, sf in sensitive_map:
    for model_name, y_pred_m in [('Baseline', y_pred), ('Fair', y_pred_fair)]:
        obs, lo, hi, verdict = bca_eod_ci(np.array(y_test), y_pred_m, sf)
        print(f"  {attr:<18} {model_name:<10} {obs:>7.3f} {lo:>8.3f} {hi:>8.3f}  {verdict}")
    print()

## 7. Conclusion

| Attribute | Baseline EOD | Fair EOD | CI Verdict | Action |
|-----------|-------------|----------|------------|--------|
| gender | _run cells above_ | | | |
| age_cohort | | | | |
| multilingual | | | | |

**Interpretation:** If the fair model CI verdict is OK/BORDERLINE for all attributes,
the re-weighting mitigation is sufficient for EU AI Act Art. 10 compliance.
If any attribute remains MITIGATE, consider ExponentiatedGradient (see `fair_wrapper.py`).